In [73]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

In [74]:
# Load original data
original_data_path = '../data/original/heart.csv'
df_original = pd.read_csv(original_data_path)

print(f"Original data shape: {df_original.shape}")
print(f"Original data columns: {df_original.columns.tolist()}")
print(df_original.head())

Original data shape: (920, 14)
Original data columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
    age  sex   cp trestbps chol fbs restecg thalach exang oldpeak slope ca  \
0  32.0  1.0  1.0       95    0   ?       0     127     0      .7     1  ?   
1  34.0  1.0  4.0      115    0   ?       ?     154     0      .2     1  ?   
2  35.0  1.0  4.0        ?    0   ?       0     130     1       ?     ?  ?   
3  36.0  1.0  4.0      110    0   ?       0     125     1       1     2  ?   
4  38.0  0.0  4.0      105    0   ?       0     166     0     2.8     1  ?   

  thal  num  
0    ?    1  
1    ?    1  
2    7    3  
3    6    1  
4    ?    2  


In [75]:
# Load and concatenate synthetic data
synthetic_dir = '../data/synthetic/'
synthetic_files = [f for f in os.listdir(synthetic_dir) if f.endswith('.csv')]
synthetic_files.sort()

dfs_synthetic = []
for file in synthetic_files:
    df_syn = pd.read_csv(os.path.join(synthetic_dir, file))
    dfs_synthetic.append(df_syn)
    print(f"Loaded {file}: shape {df_syn.shape}")

df_synthetic = pd.concat(dfs_synthetic, ignore_index=True)
print(f"\nTotal synthetic data shape: {df_synthetic.shape}")

Loaded synthetic_heart1.csv: shape (5, 14)
Loaded synthetic_heart2.csv: shape (10, 14)
Loaded synthetic_heart3.csv: shape (100, 14)
Loaded synthetic_heart4.csv: shape (500, 14)
Loaded synthetic_heart5.csv: shape (500, 14)
Loaded synthetic_heart6.csv: shape (100, 14)

Total synthetic data shape: (1215, 14)


In [76]:
# Combine original and synthetic data
df = pd.concat([df_original, df_synthetic], ignore_index=True)
print(f"Combined data shape: {df.shape}")
print(f"Original data: {len(df_original)} rows")
print(f"Synthetic data: {len(df_synthetic)} rows")

# Add a marker to identify which rows are original vs synthetic
df['is_original'] = False
df.loc[:len(df_original)-1, 'is_original'] = True

print(f"\nData distribution:")
print(df['is_original'].value_counts())

Combined data shape: (2135, 14)
Original data: 920 rows
Synthetic data: 1215 rows

Data distribution:
is_original
False    1215
True      920
Name: count, dtype: int64


In [77]:
# Replace '?' with np.nan in the dataframe
df = df.replace('?', np.nan)
print(f"Replaced '?' with np.nan")
print(f"Missing values after replacement:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Replaced '?' with np.nan
Missing values after replacement:
trestbps     59
chol         30
fbs          90
restecg       2
thalach      55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
dtype: int64


In [78]:
# Create new feature: age_thalach_ratio (safe numeric conversion & handling)
df['thalach'] = pd.to_numeric(df['thalach'], errors='coerce')
df['age_thalach_ratio'] = (df['age'] / df['thalach']).replace([np.inf, -np.inf], np.nan) + 1

print("Created 'age_thalach_ratio' — missing values:", int(df['age_thalach_ratio'].isna().sum()))

Created 'age_thalach_ratio' — missing values: 55


In [79]:
# Split data: 70% train, 30% test (test set contains only original data)
# First, separate original and synthetic data
df_original_only = df[df['is_original'] == True].copy()
df_synthetic_only = df[df['is_original'] == False].copy()

# Use all synthetic data for training
# Split original data into train and test (70/30)
# We need test to be 30% of total, using only original data
test_size = 0.3
train_size = 0.7

# To ensure test is 30% of the final split, and using only original:
# Split original: 70% train, 30% test
df_original_train, df_test = train_test_split(
    df_original_only, 
    test_size=test_size, 
    random_state=42
)

# Training data is original_train + all synthetic
df_train = pd.concat([df_original_train, df_synthetic_only], ignore_index=True)

print(f"Data split:")
print(f"Training set size: {len(df_train)} (contains original + synthetic)")
print(f"Test set size: {len(df_test)} (contains only original)")
print(f"Total: {len(df_train) + len(df_test)}")
print(f"\nTrain/Test split ratio:")
print(f"Train: {len(df_train)/(len(df_train)+len(df_test))*100:.1f}%")
print(f"Test: {len(df_test)/(len(df_train)+len(df_test))*100:.1f}%")
print(f"\nTraining set composition:")
print(f"  Original: {df_original_train.shape[0]}")
print(f"  Synthetic: {df_synthetic_only.shape[0]}")
print(f"\nTest set composition:")
print(f"  Original only: {df_test.shape[0]}")

# Reset index for both sets
df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

Data split:
Training set size: 1859 (contains original + synthetic)
Test set size: 276 (contains only original)
Total: 2135

Train/Test split ratio:
Train: 87.1%
Test: 12.9%

Training set composition:
  Original: 644
  Synthetic: 1215

Test set composition:
  Original only: 276


In [80]:
# Features with multiple categories (Nominal)
categorical_cols = [ 'cp', 'restecg', 'slope', 'thal', 'num']

# Features with only two possible values (Binary/Boolean)
bool_cols = ['sex', 'fbs', 'exang']

# Features with continuous or discrete numerical values
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'age_thalach_ratio']

## Imputation Functions

In [81]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [82]:
# Check for missing values before imputation
print("Missing values before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}\n")

Missing values before imputation:
trestbps              59
chol                  30
fbs                   90
restecg                2
thalach               55
exang                 55
oldpeak               62
slope                309
ca                   611
thal                 486
age_thalach_ratio     55
dtype: int64

Total missing values: 1814



In [83]:
def impute_categorical_column(df, target_col, categorical_cols, bool_cols, numeric_cols, random_state=42):
    """
    Imputes missing values in categorical features using machine learning approach.
    Training is based only on authentic (original) data.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    target_col : str
        Name of the categorical column with missing values to impute
    categorical_cols : list
        List of categorical column names
    bool_cols : list
        List of boolean column names
    numeric_cols : list
        List of numeric column names
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    df_imputed : pd.DataFrame
        DataFrame with imputed missing values
    model_accuracy : float
        Accuracy score of the trained classifier
    """
    
    # Step 1: Separate rows with missing and non-missing values
    df_with_missing = df[df[target_col].isna()].copy()
    df_without_missing = df[df[target_col].notna()].copy()
    
    # Filter to use only authentic data for training
    df_without_missing_authentic = df_without_missing[df_without_missing['is_original'] == True].copy()
    
    print(f"Imputing categorical column: {target_col}")
    print(f"Rows with missing values: {len(df_with_missing)}")
    print(f"Rows without missing values (all): {len(df_without_missing)}")
    print(f"Rows without missing values (authentic only): {len(df_without_missing_authentic)}")
    
    if len(df_without_missing_authentic) == 0:
        print(f"Warning: No complete authentic rows for training. Skipping imputation for {target_col}")
        return df, None
    
    # Step 2: Extract input features (X) and target variable (y) - from authentic data only
    # Exclude 'is_original' marker column
    X_train = df_without_missing_authentic.drop(columns=[target_col, 'is_original'])
    y_train = df_without_missing_authentic[target_col]
    
    # Step 3: Identify other columns with missing values
    other_missing_cols = [col for col in X_train.columns if X_train[col].isna().any()]
    
    # Step 4: Encode categorical AND boolean features
    le_dict = {}
    X_train_encoded = X_train.copy()
    
    cols_to_encode = [c for c in (categorical_cols + bool_cols) if c != target_col and c in X_train_encoded.columns]
    for col in cols_to_encode:
        le = LabelEncoder()
        valid_mask = X_train_encoded[col].notna()
        if valid_mask.any():
            X_train_encoded.loc[valid_mask, col] = le.fit_transform(
                X_train_encoded.loc[valid_mask, col].astype(str)
            )
            le_dict[col] = le
    
    # Step 5: Encode target column
    le_target = LabelEncoder()
    y_train_encoded = le_target.fit_transform(y_train.astype(str))
    
    # Step 6: Initialize IterativeImputer for other missing columns
    if other_missing_cols:
        imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=10, random_state=random_state), max_iter=10)
        imputed_array = imputer.fit_transform(X_train_encoded)
        X_train_encoded = pd.DataFrame(imputed_array, columns=X_train_encoded.columns)
    else:
        imputer = None
    
    # Step 7-8: Split and train RandomForestClassifier
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    X_tr, X_te, y_tr, y_te = sklearn_train_test_split(
        X_train_encoded, y_train_encoded, test_size=0.3, random_state=random_state
    )
    
    clf = RandomForestClassifier(n_estimators=50, random_state=random_state, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    
    # Step 9: Evaluate model
    y_pred = clf.predict(X_te)
    model_accuracy = accuracy_score(y_te, y_pred)
    print(f"Model Accuracy: {model_accuracy:.4f}")
    
    # Step 10-12: Prepare dataset with missing values and impute
    X_missing = df_with_missing.drop(columns=[target_col, 'is_original'])
    X_missing_encoded = X_missing.copy()
    
    # Encode categorical and boolean features for missing data (handle unseen labels safely)
    for col in cols_to_encode:
        if col in le_dict:
            valid_mask = X_missing_encoded[col].notna()
            if valid_mask.any():
                known_classes = set(le_dict[col].classes_)
                safe_vals = X_missing_encoded.loc[valid_mask, col].astype(str)
                known_mask = valid_mask & X_missing_encoded[col].astype(str).isin(known_classes)
                if known_mask.any():
                    X_missing_encoded.loc[known_mask, col] = le_dict[col].transform(
                        X_missing_encoded.loc[known_mask, col].astype(str)
                    )
                # Unseen labels -> NaN so the imputer handles them
                unseen_mask = valid_mask & ~X_missing_encoded[col].astype(str).isin(known_classes)
                if unseen_mask.any():
                    X_missing_encoded.loc[unseen_mask, col] = np.nan
    
    # Impute other missing columns if needed
    if imputer is not None and other_missing_cols:
        imputed_array = imputer.transform(X_missing_encoded)
        X_missing_encoded = pd.DataFrame(imputed_array, columns=X_missing_encoded.columns)
    
    # Step 13: Predict missing values
    y_missing_encoded = clf.predict(X_missing_encoded)
    
    # Step 14: Decode predictions
    y_missing = le_target.inverse_transform(y_missing_encoded)
    
    # Preserve original index for correct merge
    df_with_missing = df_with_missing.copy()
    df_with_missing[target_col] = y_missing
    df_imputed = pd.concat([df_without_missing, df_with_missing]).sort_index().reset_index(drop=True)
    
    print(f"Successfully imputed {len(df_with_missing)} missing values\n")
    
    return df_imputed, model_accuracy


In [84]:
def impute_continuous_column(df, target_col, categorical_cols, bool_cols, numeric_cols, random_state=42):
    """
    Imputes missing values in continuous (numerical) features using regression-based modeling.
    Training is based only on authentic (original) data.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    target_col : str
        Name of the continuous column with missing values to impute
    categorical_cols : list
        List of categorical column names
    bool_cols : list
        List of boolean column names
    numeric_cols : list
        List of numeric column names
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    df_imputed : pd.DataFrame
        DataFrame with imputed missing values
    metrics : dict
        Dictionary containing MSE, RMSE, and R2 score
    """
    
    # Step 1: Separate rows with missing and non-missing values
    df_with_missing = df[df[target_col].isna()].copy()
    df_without_missing = df[df[target_col].notna()].copy()
    
    # Filter to use only authentic data for training
    df_without_missing_authentic = df_without_missing[df_without_missing['is_original'] == True].copy()
    
    print(f"Imputing continuous column: {target_col}")
    print(f"Rows with missing values: {len(df_with_missing)}")
    print(f"Rows without missing values (all): {len(df_without_missing)}")
    print(f"Rows without missing values (authentic only): {len(df_without_missing_authentic)}")
    
    if len(df_without_missing_authentic) == 0:
        print(f"Warning: No complete authentic rows for training. Skipping imputation for {target_col}")
        return df, None
    
    # Step 2: Extract input features (X) and target variable (y) - from authentic data only
    # Exclude 'is_original' marker column
    X_train = df_without_missing_authentic.drop(columns=[target_col, 'is_original'])
    # Coerce target to numeric (handles any lingering object dtype from '?' replacement)
    y_train = pd.to_numeric(df_without_missing_authentic[target_col], errors='coerce')
    
    # Step 3: Identify other columns with missing values
    other_missing_cols = [col for col in X_train.columns if X_train[col].isna().any()]
    
    # Step 4: Encode categorical AND boolean features
    le_dict = {}
    X_train_encoded = X_train.copy()
    
    cols_to_encode = [c for c in (categorical_cols + bool_cols) if c != target_col and c in X_train_encoded.columns]
    for col in cols_to_encode:
        le = LabelEncoder()
        valid_mask = X_train_encoded[col].notna()
        if valid_mask.any():
            X_train_encoded.loc[valid_mask, col] = le.fit_transform(
                X_train_encoded.loc[valid_mask, col].astype(str)
            )
            le_dict[col] = le
    
    # Step 5: Initialize IterativeImputer for other missing columns
    if other_missing_cols:
        imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=10, random_state=random_state), max_iter=10)
        imputed_array = imputer.fit_transform(X_train_encoded)
        X_train_encoded = pd.DataFrame(imputed_array, columns=X_train_encoded.columns)
    else:
        imputer = None
    
    # Step 6-7: Split and train RandomForestRegressor
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    X_tr, X_te, y_tr, y_te = sklearn_train_test_split(
        X_train_encoded, y_train, test_size=0.3, random_state=random_state
    )
    
    regressor = RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)
    regressor.fit(X_tr, y_tr)
    
    # Step 8: Evaluate performance metrics
    y_pred = regressor.predict(X_te)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, y_pred)
    
    print(f"Model Performance:")
    print(f"  MSE: {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R² Score: {r2:.4f}")
    
    # Prepare dataset with missing values for imputation
    X_missing = df_with_missing.drop(columns=[target_col, 'is_original'])
    X_missing_encoded = X_missing.copy()
    
    # Encode categorical and boolean features for missing data (handle unseen labels safely)
    for col in cols_to_encode:
        if col in le_dict:
            valid_mask = X_missing_encoded[col].notna()
            if valid_mask.any():
                known_classes = set(le_dict[col].classes_)
                known_mask = valid_mask & X_missing_encoded[col].astype(str).isin(known_classes)
                if known_mask.any():
                    X_missing_encoded.loc[known_mask, col] = le_dict[col].transform(
                        X_missing_encoded.loc[known_mask, col].astype(str)
                    )
                unseen_mask = valid_mask & ~X_missing_encoded[col].astype(str).isin(known_classes)
                if unseen_mask.any():
                    X_missing_encoded.loc[unseen_mask, col] = np.nan
    
    # Impute other missing columns if needed
    if imputer is not None and other_missing_cols:
        imputed_array = imputer.transform(X_missing_encoded)
        X_missing_encoded = pd.DataFrame(imputed_array, columns=X_missing_encoded.columns)
    
    # Predict missing values
    y_missing = regressor.predict(X_missing_encoded)
    df_with_missing = df_with_missing.copy()
    df_with_missing[target_col] = y_missing
    
    # Preserve original index for correct merge
    df_imputed = pd.concat([df_without_missing, df_with_missing]).sort_index().reset_index(drop=True)
    
    print(f"Successfully imputed {len(df_with_missing)} missing values\n")
    
    metrics = {
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    }
    
    return df_imputed, metrics


In [85]:
# Check for missing values before imputation
print("Missing values before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}\n")

Missing values before imputation:
trestbps              59
chol                  30
fbs                   90
restecg                2
thalach               55
exang                 55
oldpeak               62
slope                309
ca                   611
thal                 486
age_thalach_ratio     55
dtype: int64

Total missing values: 1814



In [86]:
# Apply imputation to categorical columns
print("="*60)
print("IMPUTING CATEGORICAL COLUMNS")
print("="*60 + "\n")

df_imputed = df.copy()

for col in categorical_cols:
    if df_imputed[col].isnull().any():
        df_imputed, acc = impute_categorical_column(
            df_imputed, col, categorical_cols, bool_cols, numeric_cols
        )
    else:
        print(f"No missing values in categorical column: {col}\n")

IMPUTING CATEGORICAL COLUMNS

No missing values in categorical column: cp

Imputing categorical column: restecg
Rows with missing values: 2
Rows without missing values (all): 2133
Rows without missing values (authentic only): 918
Model Accuracy: 0.5906
Successfully imputed 2 missing values

Imputing categorical column: slope
Rows with missing values: 309
Rows without missing values (all): 1826
Rows without missing values (authentic only): 611
Model Accuracy: 0.6848
Successfully imputed 309 missing values

Imputing categorical column: thal
Rows with missing values: 486
Rows without missing values (all): 1649
Rows without missing values (authentic only): 434
Model Accuracy: 0.6489
Successfully imputed 486 missing values

No missing values in categorical column: num



In [87]:
# Apply imputation to continuous columns
print("="*60)
print("IMPUTING CONTINUOUS COLUMNS")
print("="*60 + "\n")

for col in numeric_cols:
    if df_imputed[col].isnull().any():
        df_imputed, metrics = impute_continuous_column(
            df_imputed, col, categorical_cols, bool_cols, numeric_cols
        )
    else:
        print(f"No missing values in continuous column: {col}\n")

IMPUTING CONTINUOUS COLUMNS

No missing values in continuous column: age

Imputing continuous column: trestbps
Rows with missing values: 59
Rows without missing values (all): 2076
Rows without missing values (authentic only): 861
Model Performance:
  MSE: 348.1154
  RMSE: 18.6579
  R² Score: 0.0350
Successfully imputed 59 missing values

Imputing continuous column: chol
Rows with missing values: 30
Rows without missing values (all): 2105
Rows without missing values (authentic only): 890
Model Performance:
  MSE: 7862.5512
  RMSE: 88.6710
  R² Score: 0.4106
Successfully imputed 30 missing values

Imputing continuous column: thalach
Rows with missing values: 55
Rows without missing values (all): 2080
Rows without missing values (authentic only): 865
Model Performance:
  MSE: 18.6093
  RMSE: 4.3139
  R² Score: 0.9691
Successfully imputed 55 missing values

Imputing continuous column: oldpeak
Rows with missing values: 62
Rows without missing values (all): 2073
Rows without missing values (

In [88]:
# Verify imputation results
print("="*60)
print("IMPUTATION RESULTS")
print("="*60 + "\n")

print("Missing values after imputation:")
missing_after = df_imputed.isnull().sum()
if missing_after.sum() == 0:
    print("✓ All missing values have been imputed!")
else:
    print(missing_after[missing_after > 0])

print(f"\nDataset shape: {df_imputed.shape}")
print(f"Total rows: {len(df_imputed)}")
print(f"\nDataset info:")
print(df_imputed.info())

IMPUTATION RESULTS

Missing values after imputation:
fbs      90
exang    55
dtype: int64

Dataset shape: (2135, 16)
Total rows: 2135

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2135 entries, 0 to 2134
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                2135 non-null   float64
 1   sex                2135 non-null   float64
 2   cp                 2135 non-null   float64
 3   trestbps           2135 non-null   object 
 4   chol               2135 non-null   object 
 5   fbs                2045 non-null   object 
 6   restecg            2135 non-null   object 
 7   thalach            2135 non-null   float64
 8   exang              2080 non-null   object 
 9   oldpeak            2135 non-null   object 
 10  slope              2135 non-null   object 
 11  ca                 2135 non-null   object 
 12  thal               2135 non-null   object 
 13  num                

In [89]:
# Apply imputation to boolean columns
print("="*60)
print("IMPUTING BOOLEAN COLUMNS")
print("="*60 + "\n")

for col in bool_cols:
    if df_imputed[col].isnull().any():
        df_imputed, acc = impute_categorical_column(
            df_imputed, col, categorical_cols, bool_cols, numeric_cols
        )
    else:
        print(f"No missing values in boolean column: {col}\n")

IMPUTING BOOLEAN COLUMNS

No missing values in boolean column: sex

Imputing categorical column: fbs
Rows with missing values: 90
Rows without missing values (all): 2045
Rows without missing values (authentic only): 830
Model Accuracy: 0.8394
Successfully imputed 90 missing values

Imputing categorical column: exang
Rows with missing values: 55
Rows without missing values (all): 2080
Rows without missing values (authentic only): 865
Model Accuracy: 0.7769
Successfully imputed 55 missing values



In [90]:
# Final verification of all imputation
print("="*60)
print("FINAL IMPUTATION VERIFICATION")
print("="*60 + "\n")

print("Missing values after all imputation (categorical, continuous, boolean):")
missing_final = df_imputed.isnull().sum()
if missing_final.sum() == 0:
    print("✓ All missing values have been imputed successfully!")
    print(f"\nFinal dataset shape: {df_imputed.shape}")
    print(f"Total rows: {len(df_imputed)}")
    print(f"\nAll columns imputed:")
    print(f"  - Categorical columns ({len(categorical_cols)}): {categorical_cols}")
    print(f"  - Boolean columns ({len(bool_cols)}): {bool_cols}")
    print(f"  - Numeric columns ({len(numeric_cols)}): {numeric_cols}")
else:
    print("⚠ Remaining missing values:")
    print(missing_final[missing_final > 0])

FINAL IMPUTATION VERIFICATION

Missing values after all imputation (categorical, continuous, boolean):
✓ All missing values have been imputed successfully!

Final dataset shape: (2135, 16)
Total rows: 2135

All columns imputed:
  - Categorical columns (5): ['cp', 'restecg', 'slope', 'thal', 'num']
  - Boolean columns (3): ['sex', 'fbs', 'exang']
  - Numeric columns (7): ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'age_thalach_ratio']


In [91]:
from sklearn.preprocessing import MinMaxScaler

# Re-split the fully imputed dataset into train/test
# (imputation was performed on df_imputed, so df_train/df_test must be rebuilt)
df_original_imputed = df_imputed[df_imputed['is_original'] == True].copy()
df_synthetic_imputed = df_imputed[df_imputed['is_original'] == False].copy()

df_original_train_imp, df_test = train_test_split(
    df_original_imputed, test_size=0.3, random_state=42
)
df_train = pd.concat([df_original_train_imp, df_synthetic_imputed], ignore_index=True)
df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

print(f'Train size: {len(df_train)}, Test size: {len(df_test)}')
print(f'Missing in df_train: {df_train[numeric_cols].isnull().sum().sum()}')
print(f'Missing in df_test: {df_test[numeric_cols].isnull().sum().sum()}')

scaler = MinMaxScaler()
df_train[numeric_cols] = scaler.fit_transform(df_train[numeric_cols])
df_test[numeric_cols] = scaler.transform(df_test[numeric_cols])  # use fit from train only

Train size: 1859, Test size: 276
Missing in df_train: 0
Missing in df_test: 0


## XGBoost Model Training

Train an XGBoost classifier on the entire imputed dataset using optimized hyperparameters.

In [92]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare features and target
target_col = 'num'
feature_cols = [c for c in df_train.columns if c not in [target_col, 'is_original']]

X_train = df_train[feature_cols].copy()
y_train = df_train[target_col].astype(int)
X_test = df_test[feature_cols].copy()
y_test = df_test[target_col].astype(int)

# Encode any remaining object columns
le_encoders = {}
for col in X_train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = X_test[col].astype(str).map(
        lambda val, le=le: le.transform([val])[0] if val in le.classes_ else -1
    )
    le_encoders[col] = le

# XGBoost model with optimized hyperparameters
model = xgb.XGBClassifier(
    n_estimators=570,
    max_depth=7,
    learning_rate=0.0011560224259168138,
    subsample=0.5539457134966522,
    colsample_bytree=0.5157145928433671,
    gamma=3.182052056318902,
    min_child_weight=4,
    reg_lambda=0.1082138291061399,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5145

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.56      0.69       124
           1       0.37      0.90      0.52        81
           2       0.00      0.00      0.00        31
           3       0.00      0.00      0.00        31
           4       0.00      0.00      0.00         9

    accuracy                           0.51       276
   macro avg       0.25      0.29      0.24       276
weighted avg       0.51      0.51      0.46       276

Confusion Matrix:
[[69 55  0  0  0]
 [ 8 73  0  0  0]
 [ 0 31  0  0  0]
 [ 0 31  0  0  0]
 [ 0  9  0  0  0]]


## Baseline Model: Original Data Only

Train the same XGBoost classifier using only original (non-synthetic) data for comparison.

In [93]:
# Train/test split using only original data
df_original_only_imp = df_imputed[df_imputed['is_original'] == True].copy()

df_orig_train, df_orig_test = train_test_split(
    df_original_only_imp, test_size=0.3, random_state=42
)
df_orig_train.reset_index(drop=True, inplace=True)
df_orig_test.reset_index(drop=True, inplace=True)

# Scale numeric columns (fit on original train only)
scaler_orig = MinMaxScaler()
df_orig_train[numeric_cols] = scaler_orig.fit_transform(df_orig_train[numeric_cols])
df_orig_test[numeric_cols] = scaler_orig.transform(df_orig_test[numeric_cols])

# Prepare features and target
X_train_orig = df_orig_train[feature_cols].copy()
y_train_orig = df_orig_train[target_col].astype(int)
X_test_orig = df_orig_test[feature_cols].copy()
y_test_orig = df_orig_test[target_col].astype(int)

# Encode object columns
le_encoders_orig = {}
for col in X_train_orig.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X_train_orig[col] = le.fit_transform(X_train_orig[col].astype(str))
    X_test_orig[col] = X_test_orig[col].astype(str).map(
        lambda val, le=le: le.transform([val])[0] if val in le.classes_ else -1
    )
    le_encoders_orig[col] = le

# Same XGBoost hyperparameters
model_orig = xgb.XGBClassifier(
    n_estimators=570,
    max_depth=7,
    learning_rate=0.0011560224259168138,
    subsample=0.5539457134966522,
    colsample_bytree=0.5157145928433671,
    gamma=3.182052056318902,
    min_child_weight=4,
    reg_lambda=0.1082138291061399,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model_orig.fit(X_train_orig, y_train_orig)

y_pred_orig = model_orig.predict(X_test_orig)
accuracy_orig = accuracy_score(y_test_orig, y_pred_orig)

print(f'Baseline (original data only) Accuracy: {accuracy_orig:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_orig, y_pred_orig))
print('Confusion Matrix:')
print(confusion_matrix(y_test_orig, y_pred_orig))

# Summary comparison
print('=' * 50)
print('MODEL COMPARISON SUMMARY')
print('=' * 50)
print(f'Original + Synthetic data  ->  Accuracy: {accuracy:.4f}  |  Train size: {len(X_train)}')
print(f'Original data only         ->  Accuracy: {accuracy_orig:.4f}  |  Train size: {len(X_train_orig)}')
print(f'Difference: {accuracy - accuracy_orig:+.4f}')


Baseline (original data only) Accuracy: 0.5362

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.97      0.66       124
           1       0.74      0.35      0.47        81
           2       0.00      0.00      0.00        31
           3       0.00      0.00      0.00        31
           4       0.00      0.00      0.00         9

    accuracy                           0.54       276
   macro avg       0.25      0.26      0.23       276
weighted avg       0.44      0.54      0.44       276

Confusion Matrix:
[[120   4   0   0   0]
 [ 53  28   0   0   0]
 [ 29   2   0   0   0]
 [ 27   4   0   0   0]
 [  9   0   0   0   0]]
MODEL COMPARISON SUMMARY
Original + Synthetic data  ->  Accuracy: 0.5145  |  Train size: 1859
Original data only         ->  Accuracy: 0.5362  |  Train size: 644
Difference: -0.0217
